In [1]:
# --- Notebook Test Engine: injected setup (auto-generated) ---
import os as _os
for _k in ('AWS_ACCESS_KEY_ID','AWS_SECRET_ACCESS_KEY','AWS_SESSION_TOKEN','AWS_CREDENTIAL_EXPIRATION'):
    _os.environ.pop(_k, None)
_os.environ.setdefault('AWS_DEFAULT_REGION', 'us-west-2')
_os.environ.setdefault('AWS_REGION', 'us-west-2')
_nte_role = 'arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole'
try:
    from sagemaker.core.helper import session_helper as _sh
    _sh.get_execution_role = lambda *a, **k: _nte_role
except Exception:
    pass
try:
    _nte_cf = _os.environ.get('AWS_SHARED_CREDENTIALS_FILE')
    if _nte_cf and _os.path.exists(_nte_cf):
        import configparser as _cfp, datetime as _dt
        import boto3 as _b3, botocore.session as _bcs
        from botocore.credentials import RefreshableCredentials as _RC
        _nte_prof = _os.environ.get('AWS_PROFILE', 'default')
        def _nte_load():
            _cp = _cfp.ConfigParser(); _cp.read(_nte_cf)
            _s = _cp[_nte_prof]
            _tok = _s.get('aws_session_token')
            if not _tok:
                raise RuntimeError('nte: no session token in creds file')
            return {'access_key': _s['aws_access_key_id'],
                    'secret_key': _s['aws_secret_access_key'],
                    'token': _tok,
                    'expiry_time': (_dt.datetime.now(_dt.timezone.utc)
                                    + _dt.timedelta(minutes=9)).isoformat()}
        _nte_creds = _RC.create_from_metadata(metadata=_nte_load(),
                                              refresh_using=_nte_load,
                                              method='nte-shared-file')
        def _nte_get_credentials(self, *a, **k):
            return _nte_creds
        _bcs.Session.get_credentials = _nte_get_credentials
        _b3.setup_default_session(botocore_session=_bcs.get_session())
        print('nte: refreshable shared-file creds installed')
except Exception as _e:
    print('nte: refreshable-creds shim skipped:', _e)
try:
    from sagemaker.core.resources import ModelPackageGroup as _NteMPG
    _nte_mpg_create = _NteMPG.create
    def _nte_mpg_get_or_create(*_a, **_k):
        try:
            return _nte_mpg_create(*_a, **_k)
        except Exception as _e2:
            if 'already exists' in str(_e2).lower():
                _name = _k.get('model_package_group_name') or (_a[0] if _a else None)
                return _NteMPG.get(_name)
            raise
    _NteMPG.create = staticmethod(_nte_mpg_get_or_create)
    print('nte: ModelPackageGroup.create is now get-or-create')
except Exception as _e:
    print('nte: ModelPackageGroup idempotency shim skipped:', _e)


nte: ModelPackageGroup.create is now get-or-create


# SageMaker Benchmark Evaluation - Basic Usage

This notebook demonstrates the basic user-facing flow for creating and managing benchmark evaluation jobs using the BenchmarkEvaluator with Jinja2 template-based pipeline generation.

## Step 1: Discover Available Benchmarks

Discover the benchmark properties and available options:
[Nova Model Evaluation](https://docs.aws.amazon.com/sagemaker/latest/dg/nova-model-evaluation.html)

In [2]:
# Configure AWS credentials and region
#! ada credentials update --provider=isengard --account=<> --role=Admin --profile=default --once
#! aws configure set region us-west-2

In [3]:
from sagemaker.train.evaluate import get_benchmarks, get_benchmark_properties
from rich.pretty import pprint

# Configure logging to show INFO messages
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s - %(name)s - %(message)s'
)

# Get available benchmarks
Benchmark = get_benchmarks()
pprint(list(Benchmark))

# Print properties for a specific benchmark
pprint(get_benchmark_properties(benchmark=Benchmark.MMLU))

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /opt/ml/processing/input/sm_config.yaml


[
│   <_Benchmark.MMLU: 'mmlu'>,
│   <_Benchmark.MMLU_PRO: 'mmlu_pro'>,
│   <_Benchmark.BBH: 'bbh'>,
│   <_Benchmark.GPQA: 'gpqa'>,
│   <_Benchmark.MATH: 'math'>,
│   <_Benchmark.STRONG_REJECT: 'strong_reject'>,
│   <_Benchmark.IFEVAL: 'ifeval'>,
│   <_Benchmark.MMMU: 'mmmu'>,
│   <_Benchmark.LLM_JUDGE: 'llm_judge'>
]

{
│   'modality': 'Text',
│   'description': 'Multi-task Language Understanding – Tests knowledge across 57 subjects.',
│   'metrics': ['accuracy'],
│   'strategy': 'zs_cot',
│   'subtask_available': True,
│   'subtasks': [
│   │   'abstract_algebra',
│   │   'anatomy',
│   │   'astronomy',
│   │   'business_ethics',
│   │   'clinical_knowledge',
│   │   'college_biology',
│   │   'college_chemistry',
│   │   'college_computer_science',
│   │   'college_mathematics',
│   │   'college_medicine',
│   │   'college_physics',
│   │   'computer_security',
│   │   'conceptual_physics',
│   │   'econometrics',
│   │   'electrical_engineering',
│   │   'elementary_mathematics',
│   │   'formal_logic',
│   │   'global_facts',
│   │   'high_school_biology',
│   │   'high_school_chemistry',
│   │   'high_school_computer_science',
│   │   'high_school_european_history',
│   │   'high_school_geography',
│   │   'high_school_government_and_politics',
│   │   'high_school_macroeconomics',
│   │   'high_school_mathematics',
│   │   'high_school_microeconomics',
│   │   'high_school_physics',
│   │   'high_school_psychology',
│   │   'high_school_statistics',
│   │   'high_school_us_history',
│   │   'high_school_world_history',
│   │   'human_aging',
│   │   'human_sexuality',
│   │   'international_law',
│   │   'jurisprudence',
│   │   'logical_fallacies',
│   │   'machine_learning',
│   │   'management',
│   │   'marketing',
│   │   'medical_genetics',
│   │   'miscellaneous',
│   │   'moral_disputes',
│   │   'moral_scenarios',
│   │   'nutrition',
│   │   'philosophy',
│   │   'prehistory',
│   │   'professional_accounting',
│   │   'professional_law',
│   │   'professional_medicine',
│   │   'professional_psychology',
│   │   'public_relations',
│   │   'security_studies',
│   │   'sociology',
│   │   'us_foreign_policy',
│   │   'virology',
│   │   'world_religions'
│   ]
}

## Step 2: Create BenchmarkEvaluator

Create a BenchmarkEvaluator instance with the desired benchmark. The evaluator will use Jinja2 templates to render a complete pipeline definition.

**Required Parameters:**
- `benchmark`: Benchmark type from the Benchmark enum
- `base_model`: Model ARN from SageMaker hub content
- `output_s3_location`: S3 location for evaluation outputs
- `mlflow_resource_arn`: MLflow tracking server ARN for experiment tracking

**Optional Template Fields:**
These fields are used for template rendering. If not provided, defaults will be used:
- `model_package_group`: Model package group ARN
- `source_model_package`: Source model package ARN
- `dataset`: S3 URI of evaluation dataset
- `model_artifact`: ARN of model artifact for lineage tracking (auto-inferred from source_model_package)

In [4]:
from sagemaker.train.evaluate import BenchMarkEvaluator

# Create evaluator with MMLU benchmark
# These values match our successfully tested configuration
evaluator = BenchMarkEvaluator(
    benchmark=Benchmark.MMLU,
    #subtask = "abstract_algebra" # or "all"
    model="arn:aws:sagemaker:us-west-2:674622101542:model-package/sdk-test-finetuned-models/2",
    s3_output_path="s3://notebook-test-engine-ds-674622101542-usw2/output/benchmark-eval/",
    model_package_group="arn:aws:sagemaker:us-west-2:674622101542:model-package-group/sdk-test-finetuned-models", # Optional inferred from model if model package
    base_eval_name="mmlu-eval-demo1",
    # Note: sagemaker_session is optional and will be auto-created if not provided
    # Note: region is optional and will be auto deduced using environment variables - SAGEMAKER_REGION, AWS_REGION
)

pprint(evaluator)

[07/17/26 19:29:06] INFO     Runs on sagemaker prod, region:us-west-2                                  ]8;id=1955397;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=1955398;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py#375\375]8;;\

                    INFO     Retrieved ModelPackage in region: us-west-2                    ]8;id=1955405;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/model_resolution.py\model_resolution.py]8;;\:]8;id=1955406;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/model_resolution.py#425\425]8;;\

[07/17/26 19:29:07] INFO     Found 1 MLflow apps: [('finetune-mlflow-1783049203', 'Created',  ]8;id=1955413;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/finetune_utils.py\finetune_utils.py]8;;\:]8;id=1955414;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/finetune_utils.py#213\213]8;;\
                             '3.10.1')]                                                                            

                    INFO     Resolved MLflow app:                                             ]8;id=1955420;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/finetune_utils.py\finetune_utils.py]8;;\:]8;id=1955421;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/finetune_utils.py#236\236]8;;\
                             arn:aws:sagemaker:us-west-2:674622101542:mlflow-app/app-3THB6MFI                      
                             DE4F (status: Created, version: 3.10.1)                                               

                    INFO     Resolved MLflow resource ARN:                                    ]8;id=1955428;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=1955429;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py#215\215]8;;\
                             arn:aws:sagemaker:us-west-2:674622101542:mlflow-app/app-3THB6MFI                      
                             DE4F                                                                                  

                    INFO     Model package group provided as ARN:                             ]8;id=1955435;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=1955436;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py#247\247]8;;\
                             arn:aws:sagemaker:us-west-2:674622101542:model-package-group/sdk                      
                             -test-finetuned-models                                                                

BenchMarkEvaluator(
│   region=None,
│   role=None,
│   sagemaker_session=<sagemaker.core.helper.session_helper.Session object at 0x7f66d8ae05f0>,
│   model='arn:aws:sagemaker:us-west-2:674622101542:model-package/sdk-test-finetuned-models/2',
│   base_model_name=None,
│   base_eval_name='mmlu-eval-demo1',
│   s3_output_path='s3://notebook-test-engine-ds-674622101542-usw2/output/benchmark-eval/',
│   mlflow_resource_arn='arn:aws:sagemaker:us-west-2:674622101542:mlflow-app/app-3THB6MFIDE4F',
│   mlflow_experiment_name=None,
│   mlflow_run_name=None,
│   networking=None,
│   kms_key_id=None,
│   model_package_group='arn:aws:sagemaker:us-west-2:674622101542:model-package-group/sdk-test-finetuned-models',
│   compute=None,
│   training_image=None,
│   recipe=None,
│   overrides=None,
│   benchmark=<_Benchmark.MMLU: 'mmlu'>,
│   subtasks='ALL',
│   evaluate_base_model=False
)

In [5]:
# # [Optional] BASE MODEL EVAL

# from sagemaker.train.evaluate import BenchMarkEvaluator

# # Create evaluator with MMLU benchmark
# # These values match our successfully tested configuration
# evaluator = BenchMarkEvaluator(
#     benchmark=Benchmark.MMLU,
#     model="meta-textgeneration-llama-3-2-1b-instruct",
#     s3_output_path="s3://notebook-test-engine-ds-674622101542-usw2/eval/",
#     mlflow_resource_arn="arn:aws:sagemaker:us-west-2:<>:mlflow-tracking-server/mmlu-eval-experiment",
#     # model_package_group="arn:aws:sagemaker:us-west-2:<>:model-package-group/example-name-aovqo", # Optional inferred from model if model package
#     base_eval_name="gen-qa-eval-demo",
#     # Note: sagemaker_session is optional and will be auto-created if not provided
#     # Note: region is optional and will be auto deduced using environment variables - SAGEMAKER_REGION, AWS_REGION
# )

# pprint(evaluator)

In [6]:
# # [Optional] Nova testing IAD Prod

# from sagemaker.train.evaluate import BenchMarkEvaluator

# # Create evaluator with MMLU benchmark
# # These values match our successfully tested configuration
# evaluator = BenchMarkEvaluator(
#     benchmark=Benchmark.MMLU,
#     # model="arn:aws:sagemaker:us-east-1:<>:model-package/bgrv-nova-micro-sft-lora/1",
#     model="arn:aws:sagemaker:us-east-1:<>:model-package/test-nova-finetuned-models/3",
#     s3_output_path="s3://notebook-test-engine-ds-674622101542-usw2/eval/",
#     mlflow_resource_arn="arn:aws:sagemaker:us-east-1:<>:mlflow-tracking-server/mlflow-prod-server",
#     model_package_group="arn:aws:sagemaker:us-east-1:<>:model-package-group/test-nova-finetuned-models", # Optional inferred from model if model package
#     base_eval_name="gen-qa-eval-demo",
#     region="us-east-1",
#     # Note: sagemaker_session is optional and will be auto-created if not provided
#     # Note: region is optional and will be auto deduced using environment variables - SAGEMAKER_REGION, AWS_REGION
# )

# pprint(evaluator)

### Optionally update the hyperparameters

In [7]:
pprint(evaluator.hyperparameters.to_dict())

# optionally update hyperparameters
# evaluator.hyperparameters.temperature = "0.1"

# optionally get more info on types, limits, defaults.
# evaluator.hyperparameters.get_info()


                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=1955443;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=1955444;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     Retrieved ModelPackage in region: us-west-2                    ]8;id=1955449;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/model_resolution.py\model_resolution.py]8;;\:]8;id=1955450;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/model_resolution.py#425\425]8;;\

                    INFO     Fetching evaluation override parameters for hyperparameters ]8;id=1955457;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/benchmark_evaluator.py\benchmark_evaluator.py]8;;\:]8;id=1955458;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/benchmark_evaluator.py#484\484]8;;\
                             property                                                                              

                    INFO     Fetching hub content metadata for                                  ]8;id=1955465;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/recipe_utils.py\recipe_utils.py]8;;\:]8;id=1955466;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/recipe_utils.py#226\226]8;;\
                             meta-textgeneration-llama-3-2-1b-instruct from SageMakerPublicHub                     

                    INFO     Searching for evaluation recipe with Type='Evaluation' and         ]8;id=1955472;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/recipe_utils.py\recipe_utils.py]8;;\:]8;id=1955473;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/recipe_utils.py#246\246]8;;\
                             EvaluationType='DeterministicEvaluation'                                              

                    INFO     Downloading override parameters from                               ]8;id=1955479;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/recipe_utils.py\recipe_utils.py]8;;\:]8;id=1955480;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/recipe_utils.py#271\271]8;;\
                             s3://jumpstart-cache-prod-us-west-2/recipes/open-source-eval-meta-                    
                             textgeneration-llama-3-2-1b-instruct-deterministic_override_params                    
                             _sm_jobs_v2.0.2.json                                                                  

{
│   'max_new_tokens': '2048',
│   'temperature': '0',
│   'top_k': '-1',
│   'top_p': '1.0',
│   'aggregation': '',
│   'postprocessing': 'False',
│   'max_model_len': '12000'
}

## Step 3: Run Evaluation

Start a benchmark evaluation job. The system will:
1. Build template context with all required parameters
2. Render the pipeline definition from `DETERMINISTIC_TEMPLATE` using Jinja2
3. Create or update the pipeline with the rendered definition
4. Start the pipeline execution with empty parameters (all values pre-substituted)

**What happens during execution:**
- CreateEvaluationAction: Sets up lineage tracking
- EvaluateBaseModel & EvaluateCustomModel: Run in parallel as serverless training jobs
- AssociateLineage: Links evaluation results to lineage tracking

In [8]:
# Run evaluation with configured parameters
execution = evaluator.evaluate()
pprint(execution)

print(f"\nPipeline Execution ARN: {execution.arn}")
print(f"Initial Status: {execution.status.overall_status}")

[07/17/26 19:29:08] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=1955485;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=1955486;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     Cannot simulate policies for                                  ]8;id=1955493;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=1955494;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=1955500;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=1955501;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'training' (see                                                       
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

                    INFO     Getting or creating artifact for source:                         ]8;id=1955507;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=1955508;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py#805\805]8;;\
                             arn:aws:sagemaker:us-west-2:674622101542:model-package/sdk-test-                      
                             finetuned-models/2                                                                    

                    INFO     Searching for existing artifact for model package:               ]8;id=1955514;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=1955515;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py#624\624]8;;\
                             arn:aws:sagemaker:us-west-2:674622101542:model-package/sdk-test-                      
                             finetuned-models/2                                                                    

...

[07/17/26 19:29:10] INFO     Successfully updated pipeline:                                        ]8;id=1955606;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=1955607;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py#259\259]8;;\
                             SagemakerEvaluation-BenchmarkEvaluation-629d8c99-7a50-430f-96ea-366fb                 
                             9fb4a54                                                                               

                    INFO     Starting pipeline execution: mmlu-eval-demo1-1784316550               ]8;id=1955613;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=1955614;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py#319\319]8;;\

                    INFO     Pipeline execution started:                                           ]8;id=1955620;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=1955621;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py#330\330]8;;\
                             arn:aws:sagemaker:us-west-2:674622101542:pipeline/SagemakerEvaluation                 
                             -BenchmarkEvaluation-629d8c99-7a50-430f-96ea-366fb9fb4a54/execution/e                 
                             55il836446a                                                                           

BenchmarkEvaluationExecution(
│   arn='arn:aws:sagemaker:us-west-2:674622101542:pipeline/SagemakerEvaluation-BenchmarkEvaluation-629d8c99-7a50-430f-96ea-366fb9fb4a54/execution/e55il836446a',
│   name='mmlu-eval-demo1',
│   status=PipelineExecutionStatus(overall_status='Executing', step_details=[], failure_reason=None),
│   last_modified_time=datetime.datetime(2026, 7, 17, 19, 29, 10, 494000, tzinfo=tzlocal()),
│   eval_type=<EvalType.BENCHMARK: 'benchmark'>,
│   s3_output_path='s3://notebook-test-engine-ds-674622101542-usw2/output/benchmark-eval/',
│   steps=[]
)


Pipeline Execution ARN: arn:aws:sagemaker:us-west-2:674622101542:pipeline/SagemakerEvaluation-BenchmarkEvaluation-629d8c99-7a50-430f-96ea-366fb9fb4a54/execution/e55il836446a
Initial Status: Executing


### Alternative: Override Subtasks at Runtime

For benchmarks with subtask support, you can override subtasks when calling evaluate():

In [9]:
# Override subtasks at evaluation time
# execution = mmlu_evaluator.evaluate(subtask="abstract_algebra")  # Single subtask
# execution = mmlu_evaluator.evaluate(subtask=["abstract_algebra", "anatomy"])  # Multiple subtasks

## Step 4: Monitor Execution

Check the job status and refresh as needed:

In [10]:
# Refresh status
execution.refresh()

# Display job status with step details
pprint(execution.status)

# Display individual step statuses
if execution.status.step_details:
    print("\nStep Details:")
    for step in execution.status.step_details:
        print(f"  {step.name}: {step.status}")

[07/17/26 19:29:11] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=1955626;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=1955627;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PipelineExecutionStatus(
│   overall_status='Executing',
│   step_details=[
│   │   StepDetail(
│   │   │   name='CreateEvaluationAction',
│   │   │   status='Starting',
│   │   │   start_time='2026-07-17T19:29:10.913000+00:00',
│   │   │   end_time=None,
│   │   │   display_name=None,
│   │   │   failure_reason=None,
│   │   │   job_arn=None
│   │   ),
│   │   StepDetail(
│   │   │   name='EvaluateCustomModel',
│   │   │   status='Starting',
│   │   │   start_time='2026-07-17T19:29:10.913000+00:00',
│   │   │   end_time=None,
│   │   │   display_name=None,
│   │   │   failure_reason=None,
│   │   │   job_arn=None
│   │   )
│   ],
│   failure_reason=None
)


Step Details:
  CreateEvaluationAction: Starting
  EvaluateCustomModel: Starting


## Step 5: Wait for Completion

Wait for the pipeline to complete. This provides rich progress updates in Jupyter notebooks:

In [11]:
# Wait for job completion with progress updates
# This will show a rich progress display in Jupyter
execution.wait(target_status="Succeeded", poll=5, timeout=3600)

print(f"\nFinal Status: {execution.status.overall_status}")

[07/17/26 19:57:04] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=1960518;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=1960519;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

╭─────────────────────────────────────────── Pipeline Execution Status ───────────────────────────────────────────╮
│  Evaluation Job        mmlu-eval-demo1-1784316550                                                               │
│  Links                 ]8;id=1960524;https://us-west-2.console.aws.amazon.com/cloudwatch/home?region=us-west-2#logsV2:log-groups/log-group/$252Faws$252Fsagemaker$252FTrainingJobs$3FlogStreamNameFilter$3Dpipelines-e55il836446a-EvaluateCustomModel-JiHEX61HTY\🔗 CloudWatch Logs]8;;\ | ]8;id=1960525;https://app-3THB6MFIDE4F.mlflow.sagemaker.us-west-2.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IktIS1pSUSIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNDlXNk5UVlhYaVBXanpTK05id0x4MEFicjFRbDUvcGR3TExKZXFMQ2pyRHdBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGNllVNXRiV3N3SzNSU1oza3hVWGd6WnpKSk1YSkpNbUpZV1djMlMzUkxORVV3YmtsRFpFOU5WMjByY1dadVptbFNaMVpTZEc5TGIyeERXWFpVTVVJMVVUMDlBQUVBQjJGM2N5MXJiWE1BUzJGeWJqcGhkM002YTIxek9uVnpMWGRsYzNRdE1qbzVOell4T1RNeU5UUTFOemM2YTJWNUwyVmhNbUZrTUdSa0xXRmtNV0l0TkROalppMDRaak0xTFRNMU5UWmlZVE5pWm1NeE1nQzRBUUlCQUhqRHJwaCtQMVNKcHUzZUl2YnVXUi94bGQ2THMxNjhEU3JZbXo5akp0dlBUQUZFb0Evd0FKRG8rMkd3aG9kZ2srZ0NBQUFBZmpCOEJna3Foa2lHOXcwQkJ3YWdiekJ0QWdFQU1HZ0dDU3FHU0liM0RRRUhBVEFlQmdsZ2hrZ0JaUU1FQVM0d0VRUU05MTdIcU8yZTFTcWhHaENGQWdFUWdEdUxoQjVEclNMTS8rcldBQzl3U1BnemdzQ1l3SkUyL3dxcU1kZmNLbHlIRmE2TFlhRWUxQmtsTXFROVNYZzF3TnhpOEd6eVV6a05mT243K3dJQUFCQUEwcjhEdWx2a0swcmpIdGl6VW84YjJ6QnB0S0VqU0kvbGJETENycHhvQXY3ajMwME9qWE0va0NWYi9NRmU3K1FjLy8vLy93QUFBQUVBQUFBQUFBQUFBQUFBQUFFQUFBUWI0L1VTc00vSWxsSk50OEtFSFB1OTNxT3lTTmZuMG5rdERPdkxuTHZwLytLVDBQSlpiZ3R5a2JGcktQd2RsT3NLRG9zMC90aElYcEJrd00zWk5wcnpzaGdVM290Um5pUGxmaThlQ1d2VnhUQjZRcWNkQmp4OThOTlVYai9BZXkyTlA0T2tJYWl5ODM2QnJScFdwTkJBc0lmL0I0SGlJeUVPT1BWQ2pKUDNWc255RWZvei9uaFB0S1M0M3FMKzFFVXRYOUpMdzhUVEZQQzJEMXB4elZOMDdxWXRhTk9GZ0xpaWdHdHFYb3JiRml3dE1OZGY2VnBhRzFUQmlKZHVSWXloejE5TndwbEx0ZjFTdVZFSjRTUzd6SG1YT2pHWGpyVWdJNXR2Qy9BM1FUVmViSGo4Y2I4bnEreDBkYmxOQUI4aDUvWm8zNnNGakkyU0EzMXlyZzJtSFcyc0IzbkVQNFJlOS9zTDlGSTQwOS9icE92eWFVVTRxL3NJNDVtMFNTb3JVUEFGeGRmWUxSNTA2ZDV6WmRmUWw2RmFWbnFDOXZSSXpIeTlZdmhUb3BLeWFPRHpNckhjVFNnODZzdGdneEVacHBWWTFpVEdGYnU3QmpRaTl2NFNpbDNJV0NuamRlckdWYTVyNjY1aFBBak5adnh4QTJpZ3FTaTlOYVUrelU1MGFRSkF5ZldTRTZqMXVHcTh4MW0zZU55K3F5QlR6ZkV2Yk84N3JIdXVSSDluYVREWnh1OFZOTkNRYzFSWjROWWxXQ1d1SUhwTFhnKzhqMFFNR3YySnJNTGtRZk8rTnpYK3RCVldjbWYyUU5TWHo4QlFyMlJTaWRsWHdWekJwTDFUcXNYTEVwcGhVbFJ0alBDZFVYU2YrbEF6bkZmeUcybGFidEk0QlhjN25Ud1JiWk1KQnR6c1lWTnNzeVR3M2xOSkRuRHByU2F1WHJ3NGFVVWp1K0pWL0NwaktXQjNzclpTY1FhVlF4N2Y1UElZcTk0K1BTMmNPOW9FNjFRTXRwNVNITkttWFJTODZ2L25MZUltS2hsMEFGMlFGMlVvUFUvQ2Y0VWMxZXVwVjlQVXVpVHl5a2tLSVI4Z3FtbFpoZXBiKzAzS0o3TkxJTGtKajVmR2JQemV0d2NBc013OXEzRzN4Nm1xN0FMZlJKcHJFb01FQmRTbnRobVFiWTZsQ3RvTmt5ZDkvM2xFKzlqS2hBUU1USWNUTkNZMXloK0o2M2c5QkpWRHltZnIvWVF2TWFZVWVzSG1KYXdaY3owdXZwM0VGa0Q0RUoyUElDa0xlcit0amplS0FIQThMVnRuOHdkc3FZUG5RQnFjS0Z0TEhVMHpHUkFPN0dKNVg2UHBQc0FSUHptRHVRSzd4aHRkZzkvRVkwaVhxNU5PdWFPUk9xWWMxQTgwN1hpTk5DcEk5MXNXSzNXL1RyNkV4VW5kQTVMa0NvazBMeDBEVUVuYy81RU5rRGpxL01CT0pFZFpmOEJWNVQyRVQzQ2plRW1oNkptUml4TzNhN0taMFpEOXVucVBGR3dOc29uRllRdkp6V0I2ekcrVi8xV1E1WWw1aEdtbFFPMG5WV1I2TGpTV0MxZktUNkR3dnZvQ3hYZkN3aFRaZ1RRT3JkampnWDg0QUZRRXVmNThjVnpndGFjMUpvdWZBdGcrTkR1VmdVYWNRMEFLNW1VdjlGSnMzYTQxNnRIcVdJY0N5RDRUa0J4ZmxIekUxRkhwcExSTjZtODZnQlNVL1lKemJLWnh1TTVFYnoyUmR0NlkrL0NyTGtWMnZJQWx3Zk1JNXF1SkV1UVBWdXkrUWtTUzV3MUwxQTJjTVRhNEhpK0xtTzA3N2lyUEIyZm9UcTJvUktGTFdrVVlGYUpSYTlpdGZnTlRZVlBuZWI0QVp6QmxBakVBNnA5RzNDRWdHd1R6TitmUDVyUUN2eVBGVUhNWkNCTWl5MDVScGgvVUFmNEc2ekIvVmVRQ2ZCSDJ3cWdOYnovVEFqQUh1Rnl3TGpVYkVybDVoSkprUWhMUk0xeTUvMSs4SWhLMmNYMHVTWStsbmJmL3phSjM1VVRFMTZ6dzZIN25xajQ9IiwiY2lwaGVyVGV4dCI6IkFRSUJBSGpEcnBoK1AxU0pwdTNlSXZidVdSL3hsZDZMczE2OERTclltejlqSnR2UFRBRUhSMVBwMVo2YVU5TllkUnROOWdZVUFBQUFvakNCbndZSktvWklodmNOQVFjR29JR1JNSUdPQWdFQU1JR0lCZ2txaGtpRzl3MEJCd0V3SGdZSllJWklBV1VEQkFFdU1CRUVETUJkZGdCZEVvWHdJQmhXbHdJQkVJQmI2N2llS2ZwU05aTUgySGRIVVFPUk9xN1BEaGtobU15azFsUzJjbElwUDFDeDRLa1FMN1RjVmVtR3I3WDYvWnVjZnVOVGZlYk41S0Z3aWQwWjVOYlNLN2pwS

[07/17/26 19:57:05] INFO     Final Resource Status: Succeeded                                     ]8;id=1960537;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=1960538;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py#1310\1310]8;;\


Final Status: Succeeded


## Step 6: View Results

Display the evaluation results in a formatted table:

Output Structure:

Evaluation results are stored in S3:

```
s3://your-bucket/output/
└── job_name/
    └── output/
        └── output.tar.gz
```

Extract output.tar.gz to reveal:

```
run_name/
├── eval_results/
│   ├── results_[timestamp].json
│   ├── inference_output.jsonl
│   └── details/
│       └── model/
│           └── <execution-date-time>/
│               └── details_<task_name>_#_<datetime>.parquet
└── tensorboard_results/
    └── eval/
        └── events.out.tfevents.[timestamp]
```

In [12]:
pprint(execution.s3_output_path)
# Display results in a formatted table
execution.show_results()

's3://notebook-test-engine-ds-674622101542-usw2/output/benchmark-eval/'

[07/17/26 19:57:06] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=1960543;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=1960544;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=1960549;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=1960550;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/17/26 19:57:07] INFO     S3 bucket: notebook-test-engine-ds-674622101542-usw2,        ]8;id=1960557;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py\show_results_utils.py]8;;\:]8;id=1960558;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py#129\129]8;;\
                             prefix: output/benchmark-eval                                                         

                    INFO     Extracted training job name:                                  ]8;id=1960564;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py\show_results_utils.py]8;;\:]8;id=1960565;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py#66\66]8;;\
                             pipelines-e55il836446a-EvaluateCustomModel-JiHEX61HTY from                            
                             step: EvaluateCustomModel                                                             

                    INFO     Searching for results_*.json in                              ]8;id=1960571;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py\show_results_utils.py]8;;\:]8;id=1960572;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py#152\152]8;;\
                             s3://notebook-test-engine-ds-674622101542-usw2/output/benchm                          
                             ark-eval/pipelines-e55il836446a-EvaluateCustomModel-JiHEX61H                          
                             TY/output/output/                                                                     

                    INFO     Found results file:                                          ]8;id=1960578;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py\show_results_utils.py]8;;\:]8;id=1960579;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py#171\171]8;;\
                             output/benchmark-eval/pipelines-e55il836446a-EvaluateCustomM                          
                             odel-JiHEX61HTY/output/output/eval-meta_textgeneration_llama                          
                             _3_2-g03qe/eval_results/results_2026-07-17T19-55-48.106738+0                          
                             0-00.json                                                                             

                    INFO     Using metrics from 'all' key (standard benchmark format)      ]8;id=1960585;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py\show_results_utils.py]8;;\:]8;id=1960586;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py#92\92]8;;\

                Custom Model Results                
╭────────────────────────────────┬─────────────────╮
│ Metric                         │           Value │
├────────────────────────────────┼─────────────────┤
│ mmlu_accuracy                  │          34.96% │
│ mmlu_accuracy_stderr           │          0.0353 │
╰────────────────────────────────┴─────────────────╯

╭─────────────────────────────────────────── Result Artifacts Location ───────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│  📦 Full evaluation artifacts available at:                                                                     │
│                                                                                                                 │
│  Custom Model:                                                                                                  │
│    s3://notebook-test-engine-ds-674622101542-usw2/output/benchmark-eval/pipelines-e55il836446a-EvaluateCustomM  │
│  odel-JiHEX61HTY/output/output/None/eval_results/                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Step 7: Retrieve an Existing Job

You can retrieve and inspect any existing evaluation job:

In [13]:
from sagemaker.train.evaluate import EvaluationPipelineExecution
from rich.pretty import pprint


# Get an existing job by ARN
# Replace with your actual pipeline execution ARN
existing_arn = execution.arn

# base model only example
# existing_arn = "arn:aws:sagemaker:us-west-2:<>:pipeline/SagemakerEvaluation-benchmark/execution/gdp9f4dbv2vi"
existing_execution = EvaluationPipelineExecution.get(
    arn=existing_arn,
    region="us-west-2"
)

pprint(existing_execution)
print(f"\nStatus: {existing_execution.status.overall_status}")

existing_execution.show_results()

[07/17/26 19:57:08] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=1960591;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=1960592;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     Extracted s3_output_path from training job                            ]8;id=1960598;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=1960599;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py#430\430]8;;\
                             pipelines-e55il836446a-EvaluateCustomModel-JiHEX61HTY:                                
                             s3://notebook-test-engine-ds-674622101542-usw2/output/benchmark-eval/                 

BenchmarkEvaluationExecution(
│   arn='arn:aws:sagemaker:us-west-2:674622101542:pipeline/SagemakerEvaluation-BenchmarkEvaluation-629d8c99-7a50-430f-96ea-366fb9fb4a54/execution/e55il836446a',
│   name='e55il836446a',
│   status=PipelineExecutionStatus(
│   │   overall_status='Succeeded',
│   │   step_details=[
│   │   │   StepDetail(
│   │   │   │   name='CreateEvaluationAction',
│   │   │   │   status='Succeeded',
│   │   │   │   start_time='2026-07-17T19:29:10.913000+00:00',
│   │   │   │   end_time='2026-07-17T19:29:12.062000+00:00',
│   │   │   │   display_name=None,
│   │   │   │   failure_reason=None,
│   │   │   │   job_arn=None
│   │   │   ),
│   │   │   StepDetail(
│   │   │   │   name='EvaluateCustomModel',
│   │   │   │   status='Succeeded',
│   │   │   │   start_time='2026-07-17T19:29:10.913000+00:00',
│   │   │   │   end_time='2026-07-17T19:56:59.774000+00:00',
│   │   │   │   display_name=None,
│   │   │   │   failure_reason=None,
│   │   │   │   job_arn='arn:aws:sagemaker:us-west-2:674622101542:training-job/pipelines-e55il836446a-EvaluateCustomModel-JiHEX61HTY'
│   │   │   ),
│   │   │   StepDetail(
│   │   │   │   name='AssociateLineage',
│   │   │   │   status='Succeeded',
│   │   │   │   start_time='2026-07-17T19:57:00.418000+00:00',
│   │   │   │   end_time='2026-07-17T19:57:01.518000+00:00',
│   │   │   │   display_name=None,
│   │   │   │   failure_reason=None,
│   │   │   │   job_arn=None
│   │   │   )
│   │   ],
│   │   failure_reason=None
│   ),
│   last_modified_time=datetime.datetime(2026, 7, 17, 19, 57, 1, 848000, tzinfo=tzlocal()),
│   eval_type=<EvalType.BENCHMARK: 'benchmark'>,
│   s3_output_path='s3://notebook-test-engine-ds-674622101542-usw2/output/benchmark-eval/',
│   steps=[]
)


Status: Succeeded


[07/17/26 19:57:09] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=1960604;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=1960605;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

...

                    INFO     Searching for results_*.json in                              ]8;id=1960628;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py\show_results_utils.py]8;;\:]8;id=1960629;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py#152\152]8;;\
                             s3://notebook-test-engine-ds-674622101542-usw2/output/benchm                          
                             ark-eval/pipelines-e55il836446a-EvaluateCustomModel-JiHEX61H                          
                             TY/output/output/                                                                     

[07/17/26 19:57:11] INFO     Found results file:                                          ]8;id=1960634;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py\show_results_utils.py]8;;\:]8;id=1960635;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py#171\171]8;;\
                             output/benchmark-eval/pipelines-e55il836446a-EvaluateCustomM                          
                             odel-JiHEX61HTY/output/output/eval-meta_textgeneration_llama                          
                             _3_2-g03qe/eval_results/results_2026-07-17T19-55-48.106738+0                          
                             0-00.json                                                                             

                    INFO     Using metrics from 'all' key (standard benchmark format)      ]8;id=1960640;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py\show_results_utils.py]8;;\:]8;id=1960641;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py#92\92]8;;\

                Custom Model Results                
╭────────────────────────────────┬─────────────────╮
│ Metric                         │           Value │
├────────────────────────────────┼─────────────────┤
│ mmlu_accuracy                  │          34.96% │
│ mmlu_accuracy_stderr           │          0.0353 │
╰────────────────────────────────┴─────────────────╯

╭─────────────────────────────────────────── Result Artifacts Location ───────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│  📦 Full evaluation artifacts available at:                                                                     │
│                                                                                                                 │
│  Custom Model:                                                                                                  │
│    s3://notebook-test-engine-ds-674622101542-usw2/output/benchmark-eval/pipelines-e55il836446a-EvaluateCustomM  │
│  odel-JiHEX61HTY/output/output/None/eval_results/                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [14]:
# Run evaluation with configured parameters
execution = evaluator.evaluate()
pprint(execution)

print(f"\nPipeline Execution ARN: {execution.arn}")
print(f"Initial Status: {execution.status.overall_status}")

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=1960646;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=1960647;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     Cannot simulate policies for                                  ]8;id=1960652;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=1960653;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=1960658;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=1960659;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'training' (see                                                       
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

                    INFO     Getting or creating artifact for source:                         ]8;id=1960664;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=1960665;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py#805\805]8;;\
                             arn:aws:sagemaker:us-west-2:674622101542:model-package/sdk-test-                      
                             finetuned-models/2                                                                    

                    INFO     Searching for existing artifact for model package:               ]8;id=1960670;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=1960671;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py#624\624]8;;\
                             arn:aws:sagemaker:us-west-2:674622101542:model-package/sdk-test-                      
                             finetuned-models/2                                                                    

...

                    INFO     Successfully updated pipeline:                                        ]8;id=1960748;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=1960749;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py#259\259]8;;\
                             SagemakerEvaluation-BenchmarkEvaluation-629d8c99-7a50-430f-96ea-366fb                 
                             9fb4a54                                                                               

                    INFO     Starting pipeline execution: mmlu-eval-demo1-1784318233               ]8;id=1960754;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=1960755;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py#319\319]8;;\

                    INFO     Pipeline execution started:                                           ]8;id=1960760;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=1960761;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py#330\330]8;;\
                             arn:aws:sagemaker:us-west-2:674622101542:pipeline/SagemakerEvaluation                 
                             -BenchmarkEvaluation-629d8c99-7a50-430f-96ea-366fb9fb4a54/execution/w                 
                             b8n770wo5bg                                                                           

BenchmarkEvaluationExecution(
│   arn='arn:aws:sagemaker:us-west-2:674622101542:pipeline/SagemakerEvaluation-BenchmarkEvaluation-629d8c99-7a50-430f-96ea-366fb9fb4a54/execution/wb8n770wo5bg',
│   name='mmlu-eval-demo1',
│   status=PipelineExecutionStatus(overall_status='Executing', step_details=[], failure_reason=None),
│   last_modified_time=datetime.datetime(2026, 7, 17, 19, 57, 13, 783000, tzinfo=tzlocal()),
│   eval_type=<EvalType.BENCHMARK: 'benchmark'>,
│   s3_output_path='s3://notebook-test-engine-ds-674622101542-usw2/output/benchmark-eval/',
│   steps=[]
)


Pipeline Execution ARN: arn:aws:sagemaker:us-west-2:674622101542:pipeline/SagemakerEvaluation-BenchmarkEvaluation-629d8c99-7a50-430f-96ea-366fb9fb4a54/execution/wb8n770wo5bg
Initial Status: Executing


## Step 8: List All Benchmark Evaluations

Retrieve all benchmark evaluation executions:

In [15]:
# Get all benchmark evaluations (returns iterator)
all_executions_iter = BenchMarkEvaluator.get_all(region="us-west-2")
all_executions = list(all_executions_iter)

print(f"Found {len(all_executions)} evaluation(s)\n")
for exec in all_executions[:5]:  # Show first 5
    print(f"  {exec.name}: {exec.status.overall_status}")

[07/17/26 19:57:14] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=1960766;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=1960767;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/17/26 19:57:15] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=1960772;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=1960773;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/17/26 19:57:16] INFO     Extracted s3_output_path from training job                            ]8;id=1960778;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=1960779;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py#430\430]8;;\
                             pipelines-e55il836446a-EvaluateCustomModel-JiHEX61HTY:                                
                             s3://notebook-test-engine-ds-674622101542-usw2/output/benchmark-eval/                 

                    INFO     Extracted s3_output_path from training job                            ]8;id=1960784;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=1960785;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py#430\430]8;;\
                             pipelines-04v8mee3wok0-EvaluateBaseModel-wqsA9JHfyu:                                  
                             s3://notebook-test-engine-ds-674622101542-usw2/eval-output/                           

Found 3 evaluation(s)

  wb8n770wo5bg: Executing
  e55il836446a: Succeeded
  04v8mee3wok0: Failed


## Step 9: Stop a Running Job (Optional)

You can stop a running evaluation if needed:

In [16]:
# Uncomment to stop the job
# existing_execution.stop()
# print(f"Execution stopped. Status: {execution.status.overall_status}")

## Understanding the Pipeline Structure

The rendered pipeline definition includes:

**4 Steps:**
1. **CreateEvaluationAction** (Lineage): Sets up tracking
2. **EvaluateBaseModel** (Training): Evaluates base model
3. **EvaluateCustomModel** (Training): Evaluates custom model
4. **AssociateLineage** (Lineage): Links results

**Key Features:**
- Template-based: Uses Jinja2 for flexible pipeline generation
- Parallel execution: Base and custom models evaluated simultaneously
- Serverless: No need to manage compute resources
- MLflow integration: Automatic experiment tracking
- Lineage tracking: Full traceability of evaluation artifacts

**Typical Execution Time:**
- Total: ~10-12 minutes
- Downloading phase: ~5-7 minutes (model)
- Training phase: ~3-5 minutes (running evaluation)
- Lineage steps: ~2-4 seconds each